# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:8] + "..." if hf_token else "NOT FOUND")

Token loaded: hf_DCVCv...


In [15]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Tables defined.")

Tables defined.


In [16]:
feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_ctr: (153559, 4)
feb_position: (151956, 2)


In [17]:
feb_position["feb_avg_position"].describe()

,feb_avg_position
count,151956.000000
mean,14.133842
std,15.256815
min,0.057700
25%,5.348455
50%,8.647976
75%,16.500000
max,633.000000


feb_ctr table (153,559 rows, 4 columns): content_hash_id (the page identifier), feb_clicks, feb_impressions, feb_ctr (the computed ratio).
feb_position table (151,956 rows, 2 columns): content_hash_id, feb_avg_position.

skew: one extreme (rare) value drags the average far from where most of the data actually sits. Same thing here — most pages rank reasonably (median ~8.65), but a few pages ranking terribly (position 633!) drag the average up to 14.1, making it look worse than what's typical.

In [18]:
# Merge position and CTR data together first, so we can compare them per page
signal_check_1 = feb_position.merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="inner")

# Build position buckets
def position_bucket(pos):
    if pos <= 10:
        return "top"
    elif pos <= 30:
        return "middle"
    else:
        return "poor"

signal_check_1["position_bucket"] = signal_check_1["feb_avg_position"].apply(position_bucket)

# Bucket table: average CTR per position bucket, with n (count) printed
bucket_table = signal_check_1.groupby("position_bucket").agg(
    mean_ctr=("feb_ctr", "mean"),
    n=("feb_ctr", "count")
).reindex(["top", "middle", "poor"])  # keep logical order

print(bucket_table)

                 mean_ctr      n
position_bucket                 
top              0.006083  88152
middle           0.002983  46293
poor             0.002361  17511


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal 1: CTR vs. Position (linked to FlyRank's real CTR-fix flag)

**Hypothesis:** Pages ranked well (lower average position) should get meaningfully higher
click-through rate (CTR) than poorly-ranked pages — this is the assumption behind FlyRank's
real CTR-fix flag, and matches what we already found in the starter CSV (notebook 1,
Discovery B).

**Bucket table (February 2026, warehouse data):**

| position_bucket | mean_ctr | n      |
|---|---|---|
| top (≤10)        | 0.006083 | 88,151 |
| middle (11-30)   | 0.002983 | 46,294 |
| poor (30+)       | 0.002361 | 17,511 |

**Verdict: CONFIRMED.** CTR drops as position worsens, exactly as expected — top-ranked
pages get roughly 2x the CTR of middle-ranked pages, and middle pages get somewhat higher
CTR than poorly-ranked pages. This holds across a large sample (n=151,956 total pages with
real GSC position and CTR data), giving us confidence this signal is real and safe to build
our rule on.

(Note: these raw CTR values, e.g. 0.006, appear on a different scale than the starter CSV's
CTR column, e.g. 0.15-0.35 — likely due to different scaling conventions between the two
datasets. The relative pattern across buckets, which is what matters for this verdict,
holds regardless of scale.)

In [19]:
# Rebuild March impressions (needed to recreate is_declining, same as w03)
march_impressions = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Rebuild the label: Feb vs March comparison (same 20%-drop threshold as w03)
trend_data = feb_ctr[["content_hash_id"]].merge(
    con.sql(f"""
        SELECT content_hash_id, SUM(gsc_impressions) AS feb_impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df(),
    on="content_hash_id", how="inner"
).merge(march_impressions, on="content_hash_id", how="inner")

trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print(trend_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 4)


In [20]:
# --- 3. Rebuild February features (impressions, CTR, position) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_impressions: (153559, 2)
feb_ctr: (153559, 4)
feb_position: (151956, 2)


In [21]:
# --- 5. Check feb_impressions distribution (volume signal, next step) ---
trend_data["feb_impressions"].describe()

,feb_impressions
count,134238.000000
mean,1324.283601
std,4275.575393
min,1.000000
25%,22.000000
50%,168.000000
75%,962.000000
max,203401.000000


In [22]:
# Build volume buckets using real percentiles from trend_data
def volume_bucket(imp):
    if imp <= 168:       # <= median
        return "low"
    elif imp <= 962:      # <= 75th percentile
        return "medium"
    else:
        return "high"

trend_data["volume_bucket"] = trend_data["feb_impressions"].apply(volume_bucket)

bucket_table_3 = trend_data.groupby("volume_bucket").agg(
    decline_rate=("is_declining", "mean"),
    n=("is_declining", "count")
).reindex(["low", "medium", "high"])

print(bucket_table_3)

               decline_rate      n
volume_bucket                     
low                0.224205  67135
medium             0.193890  33550
high               0.157005  33553


## Signal 2: Volume vs. Decline (linked to FlyRank's real quick-win logic)

**Hypothesis (original):** Pages with high February impressions (volume) would be MORE likely
to be declining, due to greater scrutiny/competition at scale.

**Bucket table (February 2026 impressions vs. Feb→March decline label):**

| volume_bucket | decline_rate | n      |
|---|---|---|
| low            | 0.224205     | 67,135 |
| medium         | 0.193890     | 33,550 |
| high           | 0.157005     | 33,553 |

**Verdict: OPPOSITE.** High-volume pages are actually LESS likely to be declining (15.7%)
than low-volume pages (22.4%) — a clean, monotonic pattern across a large sample
(n=134,238 total). This contradicts our original hypothesis, but is a real, usable finding:
popular, high-traffic pages appear more stable, likely because established pages have
proven staying power (similar to the pattern found by the decision tree in notebook 02).

**How this reshapes our rule:** Since high volume does not predict decline, we will NOT use
volume as a way to *find* declining pages. Instead, we use it to *prioritize among already-
declining pages*: if a page is declining, higher volume means fixing it has a bigger payoff
(more traffic at stake), making it a better "quick win" once flagged — not a standalone
signal that a page needs review in the first place.

## My Rule

**Plain words:** Flag pages that rank well (good position) but have surprisingly low CTR —
this is the CONFIRMED signal from Signal 1, indicating the listing itself (title/meta) may be
underperforming despite good visibility. Among flagged pages, prioritize those with higher
February volume — per Signal 2's OPPOSITE finding, high volume doesn't predict decline, but
it does mean a fix pays off more, since more traffic is already at stake.

**Reason code:** `ctr_underperformance_high_value` — assigned when a page has a good position
(≤10) but below-expected CTR for that tier, weighted by its traffic volume.

**Action label:** `review_title_meta` — the recommended human action: review and likely rewrite
the page's title/meta description, since the page already has visibility (good position) but
isn't converting it into clicks.

Reason code: A short, human-readable tag explaining why a specific page got flagged — like a label a reviewer can glance at to instantly understand the diagnosis, without re-reading all the numbers themselves. E.g., ctr_underperformance_high_value tells a reviewer "this page ranks well but isn't converting clicks" in one glance.

Action label: The actual task a reviewer should do next, given that diagnosis. E.g., review_title_meta tells them specifically what kind of fix to make (edit the title/meta description), not just that something is wrong.
a good avg position is earned by good content and not a good title.
A good position combined with low CTR (for that position tier) suggests the content is
relevant enough to rank well, but the title/meta description isn't compelling enough to
earn clicks once shown — a listing problem, not a content problem.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
# Merge position and CTR data (need both to check the qualifying condition)
score_data = feb_position.merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="inner")
score_data = score_data.merge(feb_impressions, on="content_hash_id", how="inner")

# Qualifying condition: good position (<=10) AND low CTR for that tier
# Using the median CTR from your "top" bucket (0.006083) as the "expected" CTR --
# anything meaningfully below that, despite good position, is underperforming.
good_position = (score_data["feb_avg_position"] <= 10).astype(int)
low_ctr = (score_data["feb_ctr"] < 0.006083).astype(int)

qualifies = good_position * low_ctr  # both must be true (binary gate, same as w01)

# Score: qualifying pages ranked by volume (higher volume = higher priority, per Signal 2)
score_data["score"] = qualifies * score_data["feb_impressions"]

score_data["reason_code"] = "ctr_underperformance_high_value"
score_data["action"] = "review_title_meta"

# Rank and check how many pages actually qualify
ranked_queue = score_data.sort_values("score", ascending=False)
print("Pages qualifying (score > 0):", (score_data["score"] > 0).sum())
ranked_queue.head(10)

Pages qualifying (score > 0): 77394


,content_hash_id,feb_avg_position,feb_ctr,feb_impressions,score,reason_code,action
18385,content_8e1334d6356668e3,4.967059,0.000010,203401.0,203401.0,ctr_underperformance_high_value,review_title_meta
13710,content_9c057b66c30a3abb,3.029402,0.000005,195648.0,195648.0,ctr_underperformance_high_value,review_title_meta
94390,content_fec55986a1868d62,3.844678,0.000000,193954.0,193954.0,ctr_underperformance_high_value,review_title_meta
18736,content_e241d6415ac9e534,2.925926,0.002443,164152.0,164152.0,ctr_underperformance_high_value,review_title_meta
110526,content_b99ea6861864dea5,3.715542,0.001699,160699.0,160699.0,ctr_underperformance_high_value,review_title_meta
85406,content_f107e54b10b43725,3.095657,0.005654,156163.0,156163.0,ctr_underperformance_high_value,review_title_meta
34540,content_acbcc847f8996314,3.907896,0.001612,148256.0,148256.0,ctr_underperformance_high_value,review_title_meta
94511,content_00d4fdf6e48a2d38,5.407108,0.004840,129333.0,129333.0,ctr_underperformance_high_value,review_title_meta
14465,content_db122b8ba22641b8,3.781584,0.003416,127941.0,127941.0,ctr_underperformance_high_value,review_title_meta
94389,content_c9f840183215651b,9.366950,0.000000,125035.0,125035.0,ctr_underperformance_high_value,review_title_meta


In [24]:
import os

os.makedirs("work/outputs", exist_ok=True)

# Only save pages that actually qualify (score > 0) -- a focused action list
# for reviewers, not a dump of all 331K+ pages including non-flagged ones.
final_queue = ranked_queue[ranked_queue["score"] > 0].reset_index(drop=True)
final_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved:", len(final_queue), "rows")

Saved: 77394 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
# row 1: Action: review_title_meta | Reason code: ctr_underperformance_high_value
# Why flagged: Ranks excellently (avg position ~5) with 203,401 February impressions, yet CTR is essentially 0 (0.00001) — a page this visible should be earning meaningfully more clicks. Strong candidate for a title/meta rewrite.
# What would make this wrong: If this page only recently moved up to position ~5, the February average position could be misleading — most of the month's impressions may have happened while it still ranked poorly, so the low CTR reflects its old position, not its current one. In that case, CTR would likely improve naturally over the next month without any title change needed.

# Row 2 — content_9c057b66c30a3abb
# Action: review_title_meta | Reason code: ctr_underperformance_high_value
# Why flagged: Ranks excellently (avg position ~3.0) with 195,648 February impressions, yet CTR is essentially 0 (0.000005) — near-total absence of clicks despite strong visibility.
# What would make this wrong: If this page only recently moved up to position ~3, the February average CTR could reflect its earlier, worse position — the low CTR may resolve naturally as the month's data catches up to its current ranking, without any title change needed.

# Row 3 — content_fec55986a1868d62
# Action: review_title_meta | Reason code: ctr_underperformance_high_value
# Why flagged: Ranks excellently (avg position ~3.8) with 193,954 February impressions, yet CTR is exactly 0 (0.000000) — literally zero clicks recorded despite very strong visibility, an even more extreme case than rows 1-2.
# What would make this wrong: Same recent-position-change risk as rows 1-2. Worth separately checking: a CTR of exactly zero (not just very low) is also consistent with a data or tracking issue (e.g., click tracking failing for this specific page) rather than a genuine listing problem — worth a quick sanity check before assuming it's purely a title issue.

# Row 4 — content_e241d6415ac9e534 (position 2.93, CTR 0.002443, 164,152 impressions)
# Action: review_title_meta | Reason code: ctr_underperformance_high_value
# Why flagged: Exceptional position (~2.9) but CTR far below the top-tier average (0.0024 vs expected ~0.006) — strong visibility not converting to clicks.
# What would make this wrong: Same recent-position-change risk — if the page only recently reached this top position, February's average CTR may reflect a worse historical position.

# Row 5 — content_b99ea6861864dea5 (position 3.72, CTR 0.001699, 160,699 impressions)
# Same structure as Row 4 — strong position, CTR well below expected average, same recent-change caveat applies.

# Row 6 — content_f107e54b10b43725 (position 3.10, CTR 0.005654, 156,163 impressions)
# Action: review_title_meta | Reason code: ctr_underperformance_high_value
# Why flagged: Technically below our 0.006083 threshold, but only marginally (0.0057 vs 0.0061) — CTR is essentially in line with what's expected for this position tier.
# What would make this wrong: This may be a false positive — the gap between this page's CTR and the expected average is small enough that it could just be normal variation, not a genuine listing problem. The rule's binary threshold doesn't distinguish "barely below average" from "dramatically below average," which is a real limitation worth naming.

# Row 7 — content_acbcc847f8996314 (position 3.91, CTR 0.001612, 148,256 impressions)
# Same structure as Row 4 — strong position, low CTR, same caveat.

# Row 8 — content_00d4fdf6e48a2d38 (position 5.41, CTR 0.004840, 129,333 impressions)
# Same reasoning as Row 6 — CTR (0.0048) is reasonably close to the 0.0061 expected average; likely a weaker, possibly false-positive flag from the same binary-threshold limitation.

# Row 9 — content_db122b8ba22641b8 (position 3.78, CTR 0.003416, 127,941 impressions)
# Same structure, though CTR here (0.0034) is somewhat closer to expected than rows 4/5/7 — still a real gap, worth flagging, but a slightly less extreme case.

# Row 10 — content_c9f840183215651b (position 9.37, CTR 0.000000, 125,035 impressions)
# Why flagged: Position is at the edge of "top" tier (9.37, near the ≤10 cutoff) with CTR at exactly 0 despite 125K impressions.
# What would make this wrong: Two possible explanations here: (1) the recent-position-change risk, same as others, or (2) since position 9.37 is near the boundary of our "good position" threshold, this page may be borderline — if its true position is closer to 10-11 on most days, the "expected high CTR" assumption weakens, since position 10 already sees a meaningful CTR drop-off (per Discovery B / Signal 1's own bucket pattern).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks summary

Three of the top 10 are weaker flags than the rest:

- **Rows 6 and 8** (CTR 0.0057 and 0.0048): both sit very close to the expected top-tier
  average CTR (0.0061) — likely false positives. Our binary threshold doesn't distinguish
  "barely below average" from "dramatically below average" (like rows 1-3's near-zero CTR),
  so these two may not actually need review.

- **Row 10** (position 9.37, at the edge of our "good position" ≤10 cutoff): flattens a real
  difference — a page at position 9 shouldn't be held to the same CTR expectation as a page
  at position 2, but our binary top/middle/poor bucketing treats them identically. This is a
  limitation of the rule's simplicity, not necessarily evidence the page needs review.

The remaining 7 picks (rows 1-5, 7, 9) show a consistent, strong pattern: excellent position,
CTR far below expected, high impressions — genuinely solid candidates for review.

In [26]:
# **Note on row 10:** Unlike rows 6/8 (CTR close to expected, likely false positives), row 10's
# weakness is different — its position (9.37) sits at the very edge of our "good position" (≤10)
# threshold, so it shouldn't be held to the same CTR expectation as a page ranking at position 2.
# Our binary top/middle/poor bucketing doesn't distinguish position 2 from position 9 within the
# same "top" tier, flattening a real difference — a limitation of the rule's simplicity, not
# necessarily evidence the page needs review.

In [27]:
# LEAKAGE CHECK: explicitly list every column used to build the score,
# and confirm none of them originate from March or from the label itself.

print("Columns used in the score:")
print(score_data.columns.tolist())

print("\nColumns explicitly used in the qualifying/scoring logic:")
print("- feb_avg_position  (from February fact_daily, before March)")
print("- feb_ctr           (from February fact_daily, before March)")
print("- feb_impressions   (from February fact_daily, before March)")

print("\nConfirming NONE of these appear anywhere in march_impressions or trend_data's label formula:")
print("march_impressions columns:", march_impressions.columns.tolist())
print("is_declining formula uses: march_impressions vs feb_impressions (label only, never fed as a feature)")

# Direct proof: is_declining was NEVER merged into score_data or used in qualifies/score
assert "is_declining" not in score_data.columns, "LEAK: is_declining found in score_data!"
assert "march_impressions" not in score_data.columns, "LEAK: march_impressions found in score_data!"
print("\nPASSED: neither is_declining nor march_impressions appear in score_data. No leakage.")

Columns used in the score:
['content_hash_id', 'feb_avg_position', 'feb_ctr', 'feb_impressions', 'score', 'reason_code', 'action']

Columns explicitly used in the qualifying/scoring logic:
- feb_avg_position  (from February fact_daily, before March)
- feb_ctr           (from February fact_daily, before March)
- feb_impressions   (from February fact_daily, before March)

Confirming NONE of these appear anywhere in march_impressions or trend_data's label formula:
march_impressions columns: ['content_hash_id', 'march_impressions']
is_declining formula uses: march_impressions vs feb_impressions (label only, never fed as a feature)

PASSED: neither is_declining nor march_impressions appear in score_data. No leakage.


In [28]:
assert "is_declining" not in final_queue.columns, "LEAK: is_declining found in final_queue!"
assert "march_impressions" not in final_queue.columns, "LEAK: march_impressions found in final_queue!"
print("PASSED: final_queue (the actual saved CSV) is also leakage-free.")

PASSED: final_queue (the actual saved CSV) is also leakage-free.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.